# Simulate template stimulus responses

Loads a template waveform from `data/template_waveforms.pkl` (generated by `26_05_26_generate_template_waveforms.ipynb`) and runs multi-trial simulations.

**Run the generate notebook first** if `data/template_waveforms.pkl` does not exist.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from designer_waveform.models import RandomEINetwork, load_config

In [ ]:
# ── Choose which template to simulate ────────────────────────────────────
# Options: 'pulse_25ms', 'pulse_135ms', 'train_25hz_1s'
TEMPLATE_NAME = 'pulse_25ms'

# ── PSTH bin size for analysis and display ───────────────────────────────
BIN_SIZE_MS = 10.0

# ── Multi-run settings ───────────────────────────────────────────────────
N_RUNS    = 20
SEED_BASE = 1000

VARY_INIT_V       = True
VARY_CONNECTIVITY = True
VARY_WEIGHTS      = True

# ── Network settings ─────────────────────────────────────────────────────
N_EXC     = 2000
N_INH     = 500
T_PRE_MS  = 200.0
T_POST_MS = 100.0

In [ ]:
TEMPLATES_PATH = Path('../data/template_waveforms.pkl')
with open(TEMPLATES_PATH, 'rb') as f:
    templates = pickle.load(f)

if TEMPLATE_NAME not in templates:
    raise KeyError(f'TEMPLATE_NAME={TEMPLATE_NAME!r} not found. Available: {list(templates)}')

template    = templates[TEMPLATE_NAME]
waveform    = template['waveform']
STIM_DUR_MS = float(template['stim_dur_ms'])
description = template['description']

print(f'Template   : {TEMPLATE_NAME}')
print(f'Description: {description}')
print(f'Waveform   : {waveform}')
print(f'Stim window: {STIM_DUR_MS:.0f} ms')
print(f'Bin size   : {BIN_SIZE_MS:.0f} ms')

In [ ]:
CONFIG_PATH = Path('..') / 'configs' / 'random_ei.json'
cfg = load_config(CONFIG_PATH)
cfg.N_exc       = N_EXC
cfg.N_inh       = N_INH
cfg.t_pre_ms    = T_PRE_MS
cfg.t_post_ms   = T_POST_MS
cfg.t_stim_ms   = STIM_DUR_MS
cfg.psth_bin_ms = BIN_SIZE_MS

model = RandomEINetwork(cfg)
print(f'Model built.  Opsin mean: {model._stim_dist_pA.mean():.1f} pA, '
      f'frac zero: {(model._stim_dist_pA == 0).mean():.3f}')

## Waveform preview

In [ ]:
_t_plot = np.linspace(0, STIM_DUR_MS, int(STIM_DUR_MS / 0.1))

fig, ax = plt.subplots(figsize=(max(7, STIM_DUR_MS / 80), 2.5))
ax.fill_between(_t_plot, 0, waveform(_t_plot), alpha=0.35, color='steelblue')
ax.plot(_t_plot, waveform(_t_plot), color='steelblue', lw=1.5)
ax.set_xlim(0, STIM_DUR_MS)
ax.set_ylim(-0.05, 1.15)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Envelope amplitude')
ax.set_title(f'{TEMPLATE_NAME} — {description}')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

## Multi-run simulation

In [ ]:
_vary_str = ', '.join(
    s for s, v in [('init_v', VARY_INIT_V), ('connectivity', VARY_CONNECTIVITY),
                   ('weights', VARY_WEIGHTS)] if v
) or 'none'
print(f'Running {N_RUNS} simulations (varying: {_vary_str})...')

_all_psth  = []
_all_fine  = []
_spike_data_one = None   # store one run for raster

_fine_bin_ms   = max(1.0, BIN_SIZE_MS / 2.0)   # finer bin for display
_fine_edges    = np.arange(0, STIM_DUR_MS + _fine_bin_ms, _fine_bin_ms)
_fine_t        = 0.5 * (_fine_edges[:-1] + _fine_edges[1:])

for _i in range(N_RUNS):
    _r = model.run(waveform, seed=SEED_BASE + _i,
                   vary_init_v=VARY_INIT_V,
                   vary_connectivity=VARY_CONNECTIVITY,
                   vary_weights=VARY_WEIGHTS)
    _all_psth.append(_r['psth_exc'] / (BIN_SIZE_MS / 1000.0))

    _exc_m  = _r['spike_indices'] < cfg.N_exc
    _et     = _r['spike_times_ms'][_exc_m] - cfg.t_pre_ms
    _win    = (_et >= 0) & (_et <= STIM_DUR_MS)
    _cnt, _ = np.histogram(_et[_win], bins=_fine_edges)
    _all_fine.append(_cnt / cfg.N_exc / (_fine_bin_ms / 1000.0))

    if _i == 0:
        _spike_data_one = {
            'times':   _r['spike_times_ms'][_exc_m][_win] - 0,
            'indices': _r['spike_indices'][_exc_m][_win],
            'offset':  cfg.t_pre_ms,
        }
    if (_i + 1) % 5 == 0:
        print(f'  {_i + 1}/{N_RUNS}')

_psth_arr  = np.stack(_all_psth)
_fine_arr  = np.stack(_all_fine)
t_psth_ms  = model.run(waveform)['t_psth_ms']

mean_hz    = _psth_arr.mean(0)
sem_hz     = _psth_arr.std(0) / np.sqrt(N_RUNS)
mean_fine  = _fine_arr.mean(0)
sem_fine   = _fine_arr.std(0) / np.sqrt(N_RUNS)

print('Done.')

## Results

In [ ]:
_fig_w = max(13, STIM_DUR_MS / 30)
fig, axes = plt.subplots(1, 2, figsize=(_fig_w, 4))

# ── Left: optimisation-resolution lines ──────────────────────────────────
ax = axes[0]
for _run_hz in _psth_arr:
    ax.plot(t_psth_ms, _run_hz, color='steelblue', lw=0.7, alpha=0.2)
ax.fill_between(t_psth_ms, mean_hz - sem_hz, mean_hz + sem_hz,
                color='steelblue', alpha=0.35)
ax.plot(t_psth_ms, mean_hz, color='steelblue', lw=2,
        label=f'Mean \u00b1 SEM (n={N_RUNS})')
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'{TEMPLATE_NAME} | {BIN_SIZE_MS:.0f} ms bins  [varying: {_vary_str}]')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

# ── Right: fine-bin lines ─────────────────────────────────────────────────
ax = axes[1]
for _run_hz in _fine_arr:
    ax.plot(_fine_t, _run_hz, color='steelblue', lw=0.7, alpha=0.2)
ax.fill_between(_fine_t, mean_fine - sem_fine, mean_fine + sem_fine,
                color='steelblue', alpha=0.35)
ax.plot(_fine_t, mean_fine, color='steelblue', lw=2,
        label=f'Mean \u00b1 SEM (n={N_RUNS})')
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'{TEMPLATE_NAME} | {_fine_bin_ms:.0f} ms bins')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
N_RASTER = 200   # neurons to show

_sd  = _spike_data_one
_st  = _sd['times'] - _sd['offset']   # relative to stim onset
_si  = _sd['indices']
_win = (_st >= 0) & (_st <= STIM_DUR_MS)
_rm  = (_si < N_RASTER) & _win

fig, axes = plt.subplots(2, 1, figsize=(max(9, STIM_DUR_MS / 30), 6),
                          gridspec_kw={'height_ratios': [1, 2]})

# Waveform
ax = axes[0]
ax.fill_between(_t_plot, 0, waveform(_t_plot), alpha=0.3, color='steelblue')
ax.plot(_t_plot, waveform(_t_plot), color='steelblue', lw=1.2)
ax.set_xlim(0, STIM_DUR_MS)
ax.set_ylabel('Envelope')
ax.set_title(f'Raster — {N_RASTER} exc neurons, run 0  [{TEMPLATE_NAME}]')
ax.tick_params(bottom=False, labelbottom=False)
ax.spines[['top', 'right', 'bottom']].set_visible(False)

# Raster
ax = axes[1]
ax.scatter(_st[_rm], _si[_rm], s=1.5, color='k', alpha=0.5, linewidths=0)
ax.set_xlim(0, STIM_DUR_MS)
ax.set_ylim(0, N_RASTER)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Neuron index')
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
OUTPUT_DIR = Path('../results') / f'26_05_26_template_responses'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_save = {
    'template_name':  TEMPLATE_NAME,
    'description':    description,
    'stim_dur_ms':    STIM_DUR_MS,
    'bin_size_ms':    BIN_SIZE_MS,
    'fine_bin_ms':    _fine_bin_ms,
    'n_runs':         N_RUNS,
    'seed_base':      SEED_BASE,
    'waveform':       waveform,
    't_psth_ms':      t_psth_ms,
    'runs_hz':        _psth_arr,
    'mean_hz':        mean_hz,
    'sem_hz':         sem_hz,
    'fine_t_ms':      _fine_t,
    'fine_mean_hz':   mean_fine,
    'fine_sem_hz':    sem_fine,
}
_save_path = OUTPUT_DIR / f'{TEMPLATE_NAME}_bin{BIN_SIZE_MS:.0f}ms.pkl'
with open(_save_path, 'wb') as f:
    pickle.dump(_save, f)
print(f'Saved to {_save_path}')